In [1]:
import pandas as pd

# Load datasets
train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")

# Check shapes
print("Train:", train.shape)
print("Validation:", val.shape)

Train: (139780, 14)
Validation: (25676, 14)


In [2]:
for name, df in [("Train", train), ("Validation", val)]:

    keys = ["person", "night", "epoch"]

    # Check chronological order
    sorted_df = df.sort_values(keys)

    is_sorted = df[keys].reset_index(drop=True).equals(
        sorted_df[keys].reset_index(drop=True)
    )

    # Check duplicate epochs
    duplicates = df.duplicated(keys).sum()

    # Check missing epochs within each night
    epoch_diff = sorted_df.groupby(
        ["person", "night"]
    )["epoch"].diff()

    gaps = (epoch_diff > 1).sum()

    print(f"\n{name}")
    print("Already sorted:", is_sorted)
    print("Duplicate epochs:", duplicates)
    print("Epoch gaps:", gaps)


Train
Already sorted: True
Duplicate epochs: 0
Epoch gaps: 315

Validation
Already sorted: True
Duplicate epochs: 0
Epoch gaps: 123


In [3]:
for name, df in [("Train", train), ("Validation", val)]:

    diff = df.groupby(["person", "night"])["epoch"].diff()

    gaps = diff[diff > 1]

    print(f"\n{name}")
    print("Gap locations:", len(gaps))
    print("Total missing epochs:", int((gaps - 1).sum()))
    print("Largest gap (epochs):", int(gaps.max() - 1))


Train
Gap locations: 315
Total missing epochs: 2223
Largest gap (epochs): 159

Validation
Gap locations: 123
Total missing epochs: 557
Largest gap (epochs): 95


In [4]:
for name, df in [("Train", train), ("Validation", val)]:

    # Find where a new segment should start
    diff = df.groupby(["person", "night"])["epoch"].diff()

    # Start a new segment at each night or gap
    new_segment = diff != 1

    # Assign segment numbers
    df["segment"] = new_segment.cumsum()

    print(name)
    print("Continuous segments:", df["segment"].nunique())

Train
Continuous segments: 487
Validation
Continuous segments: 157


In [5]:
for name, df in [("Train", train), ("Validation", val)]:

    summary = df.groupby("segment").agg(
        rows=("epoch", "size"),
        people=("person", "nunique"),
        nights=("night", "nunique")
    )

    # Check for problems
    mixed_people = (summary["people"] > 1).sum()
    mixed_nights = (summary["nights"] > 1).sum()

    epoch_diff = df.groupby("segment")["epoch"].diff()
    broken_segments = (epoch_diff.dropna() != 1).sum()

    print(f"\n{name}")
    print("Mixed participants:", mixed_people)
    print("Mixed nights:", mixed_nights)
    print("Non-consecutive epochs:", broken_segments)
    print("Segments shorter than 5 epochs:",
          (summary["rows"] < 5).sum())


Train
Mixed participants: 0
Mixed nights: 0
Non-consecutive epochs: 0
Segments shorter than 5 epochs: 82

Validation
Mixed participants: 0
Mixed nights: 0
Non-consecutive epochs: 0
Segments shorter than 5 epochs: 47


In [6]:
for name, df in [("Train", train), ("Validation", val)]:

    df["hr_mean_roll5"] = df.groupby("segment")["hr_mean"].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )

    print(f"\n{name}")
    print("Missing values:", df["hr_mean_roll5"].isna().sum())

# Display first 8 rows
display(train[["epoch", "hr_mean", "hr_mean_roll5"]].head(8))


Train
Missing values: 0

Validation
Missing values: 0


,epoch,hr_mean,hr_mean_roll5
0,0,64.333333,64.333333
1,1,69.333333,66.833333
2,2,74.000000,69.222222
3,3,76.666667,71.083333
4,4,72.000000,71.266667
5,5,70.000000,72.400000
6,6,69.600000,72.453333
7,7,67.500000,71.153333


In [7]:
display(train[[
    "person", "night", "epoch",
    "hr_mean", "hr_mean_roll5"
]].head(8))

,person,night,epoch,hr_mean,hr_mean_roll5
0,Bidslab00,1,0,64.333333,64.333333
1,Bidslab00,1,1,69.333333,66.833333
2,Bidslab00,1,2,74.000000,69.222222
3,Bidslab00,1,3,76.666667,71.083333
4,Bidslab00,1,4,72.000000,71.266667
5,Bidslab00,1,5,70.000000,72.400000
6,Bidslab00,1,6,69.600000,72.453333
7,Bidslab00,1,7,67.500000,71.153333


In [8]:
for name, df in [("Train", train), ("Validation", val)]:

    df["motion_mean_roll5"] = df.groupby("segment")["motion_mean"].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )

    print(f"\n{name}")
    print("Missing values:", df["motion_mean_roll5"].isna().sum())

# Check results
display(train[[
    "epoch", "motion_mean", "motion_mean_roll5"
]].head(8))


Train
Missing values: 0

Validation
Missing values: 0


,epoch,motion_mean,motion_mean_roll5
0,0,1.005263,1.005263
1,1,1.016832,1.011048
2,2,1.048849,1.023648
3,3,1.052265,1.030802
4,4,1.032463,1.031134
5,5,1.027740,1.035630
6,6,1.018874,1.036038
7,7,1.016141,1.029496


In [9]:
for name, df in [("Train", train), ("Validation", val)]:

    df["hr_change"] = df.groupby("segment")["hr_mean"].diff()

    # First epoch of each segment has no previous measurement
    df["hr_change"] = df["hr_change"].fillna(0)

    print(f"\n{name}")
    print("Missing values:", df["hr_change"].isna().sum())

display(train[[
    "epoch", "hr_mean", "hr_change"
]].head(8))


Train
Missing values: 0

Validation
Missing values: 0


,epoch,hr_mean,hr_change
0,0,64.333333,0.000000
1,1,69.333333,5.000000
2,2,74.000000,4.666667
3,3,76.666667,2.666667
4,4,72.000000,-4.666667
5,5,70.000000,-2.000000
6,6,69.600000,-0.400000
7,7,67.500000,-2.100000


In [10]:
for name, df in [("Train", train), ("Validation", val)]:

    df["hr_mean_roll_std5"] = df.groupby("segment")["hr_mean"].transform(
        lambda x: x.rolling(window=5, min_periods=2).std()
    )

    # First epoch has no variability history
    df["hr_mean_roll_std5"] = df["hr_mean_roll_std5"].fillna(0)

    print(f"\n{name}")
    print("Missing values:", df["hr_mean_roll_std5"].isna().sum())

display(train[[
    "epoch", "hr_mean", "hr_mean_roll_std5"
]].head(8))


Train
Missing values: 0

Validation
Missing values: 0


,epoch,hr_mean,hr_mean_roll_std5
0,0,64.333333,0.000000
1,1,69.333333,3.535534
2,2,74.000000,4.834291
3,3,76.666667,5.425420
4,4,72.000000,4.716402
5,5,70.000000,3.003701
6,6,69.600000,2.937270
7,7,67.500000,3.471439


In [11]:
for name, df in [("Train", train), ("Validation", val)]:

    df["motion_mean_roll_std5"] = df.groupby("segment")["motion_mean"].transform(
        lambda x: x.rolling(window=5, min_periods=2).std()
    )

    # First epoch of each segment has no previous history
    df["motion_mean_roll_std5"] = df["motion_mean_roll_std5"].fillna(0)

    print(f"\n{name}")
    print("Missing values:", df["motion_mean_roll_std5"].isna().sum())

display(train[[
    "epoch", "motion_mean", "motion_mean_roll_std5"
]].head(8))


Train
Missing values: 0

Validation
Missing values: 0


,epoch,motion_mean,motion_mean_roll_std5
0,0,1.005263,0.000000
1,1,1.016832,0.008181
2,2,1.048849,0.022578
3,3,1.052265,0.023336
4,4,1.032463,0.020224
5,5,1.027740,0.014808
6,6,1.018874,0.014175
7,7,1.016141,0.014331


In [12]:
# Original features
base_features = [
    "hr_mean", "hr_std", "hr_min", "hr_max",
    "motion_mean", "motion_std", "motion_min", "motion_max"
]

# New temporal features
temporal_features = [
    "hr_mean_roll5",
    "motion_mean_roll5",
    "hr_change",
    "hr_mean_roll_std5",
    "motion_mean_roll_std5"
]

# Combine features
features = base_features + temporal_features

# Prepare datasets
X_train = train[features]
y_train = train["stage_4"]

X_val = val[features]
y_val = val["stage_4"]

# Check shapes
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

print("Number of features:", len(features))

print("Train missing values:", X_train.isna().sum().sum())
print("Validation missing values:", X_val.isna().sum().sum())

X_train: (139780, 13)
X_val: (25676, 13)
Number of features: 13
Train missing values: 0
Validation missing values: 0


In [13]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit on training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same scaler to validation data
X_val_scaled = scaler.transform(X_val)

# Verify
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)

print("\nTraining feature means:")
print(X_train_scaled.mean(axis=0).round(2))

print("\nTraining feature standard deviations:")
print(X_train_scaled.std(axis=0).round(2))

Train: (139780, 13)
Validation: (25676, 13)

Training feature means:
[ 0. -0. -0.  0. -0. -0. -0. -0.  0. -0. -0. -0.  0.]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [14]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

from keras import Sequential, Input
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.utils import set_random_seed

set_random_seed(42)

# Calculate soft class weights
classes = np.sort(y_train.unique())

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

soft_weights = {
    int(c): float(np.sqrt(w))
    for c, w in zip(classes, weights)
}

print("Soft Class Weights:", soft_weights)

# Build neural network
model = Sequential([
    Input(shape=(13,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(4, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

I0000 00:00:1794330099.890286   30079 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794330099.990965   30079 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794330101.637377   30079 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Soft Class Weights: {0: 1.5295399750627603, 1: 0.7522705172835694, 2: 1.148662739989314, 3: 0.9770224004489462}


W0000 00:00:1794330102.915613   30079 gpu_device.cc:2453] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1794330102.919181   30079 gpu_device.cc:2453] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1794330103.081734   30079 gpu_device.cc:2036] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9217 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:02:00.0, compute capability: 12.0a
E0000 00:00:1794330103.148566   30079 cuda_executor.cc:1365] [0] Failed to allocate device memory of 9.00GiB (9664724992 bytes): RESOURCE_EXHAUSTED: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:45
external/xla/xla/

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,108 (12.14 KB)

 Trainable params: 3,108 (12.14 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=256,
    class_weight=soft_weights,
    callbacks=[early_stop]
)

Epoch 1/100


I0000 00:00:1794330106.562041   30191 service.cc:154] XLA service 0x7653d8525720 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1794330106.562096   30191 service.cc:170]   StreamExecutor [0]: NVIDIA GeForce RTX 5070, Compute Capability 12.0a (Driver: 13.3.0[610.88.0]; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1794330106.588146   30191 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1794330107.445208   30281 subprocess_compilation.cc:349] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_1_4', 32 bytes spill stores, 32 bytes spill loads

I0000 00:00:1794330108.225517   30273 subprocess_compilation.cc:349] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_22', 32 bytes spill stores, 32 bytes spill loads



 30/547 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3888 - loss: 1.2882  

I0000 00:00:1794330110.471221   30191 cuda_dnn.cc:442] Loaded cuDNN version 92700
I0000 00:00:1794330110.609347   30191 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


547/547 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4429 - loss: 1.1784

I0000 00:00:1794330113.600409   31139 subprocess_compilation.cc:349] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 8 bytes spill stores, 8 bytes spill loads



547/547 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.4429 - loss: 1.1784 - val_accuracy: 0.4702 - val_loss: 1.1482
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4594 - loss: 1.1354 - val_accuracy: 0.4669 - val_loss: 1.1436
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4636 - loss: 1.1276 - val_accuracy: 0.4652 - val_loss: 1.1439
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4665 - loss: 1.1235 - val_accuracy: 0.4694 - val_loss: 1.1373
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4680 - loss: 1.1199 - val_accuracy: 0.4704 - val_loss: 1.1354
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4714 - loss: 1.1175 - val_accuracy: 0.4699 - val_loss: 1.1320
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4727 - loss: 1.1148 - val_accuracy: 0.4723 - val_loss: 1.1314
Epoch 8/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4733 - loss: 1.1126 - val_accuracy: 0.4753

In [16]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Predict validation data
y_prob = model.predict(X_val_scaled)
y_pred = np.argmax(y_prob, axis=1)

# Calculate metrics
print("Validation Accuracy:",
      round(accuracy_score(y_val, y_pred) * 100, 2), "%")

print("Macro F1:",
      round(f1_score(y_val, y_pred, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred,
    target_names=["Wake", "Light", "Deep", "REM"]
))

803/803 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
Validation Accuracy: 49.05 %
Macro F1: 0.4759

Classification Report:
              precision    recall  f1-score   support

        Wake       0.48      0.47      0.48      2760
       Light       0.53      0.54      0.54     11731
        Deep       0.47      0.42      0.44      4687
         REM       0.44      0.46      0.45      6498

    accuracy                           0.49     25676
   macro avg       0.48      0.47      0.48     25676
weighted avg       0.49      0.49      0.49     25676



In [17]:
import joblib

# Save model
model.save("sleep_nn_temporal.keras")

# Save scaler
joblib.dump(scaler, "sleep_temporal_scaler.pkl")

# Save feature names and order
joblib.dump(features, "sleep_temporal_features.pkl")

print("Model, scaler, and feature list saved!")

Model, scaler, and feature list saved!


In [18]:
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

results = pd.DataFrame({
    "person": val["person"],
    "actual": y_val,
    "predicted": y_pred
})

summary = []

for person, group in results.groupby("person"):
    summary.append({
        "person": person,
        "epochs": len(group),
        "accuracy": accuracy_score(
            group["actual"], group["predicted"]
        ),
        "macro_f1": f1_score(
            group["actual"],
            group["predicted"],
            labels=[0, 1, 2, 3],
            average="macro",
            zero_division=0
        )
    })

summary_df = pd.DataFrame(summary)

display(summary_df.round(3))

,person,epochs,accuracy,macro_f1
0,Bidslab09,3544,0.500,0.488
1,Bidslab11,4852,0.457,0.454
2,Bidslab16,5536,0.545,0.512
3,Bidslab31,4441,0.466,0.441
4,Bidslab38,1992,0.456,0.396
5,Bidslab49,1632,0.406,0.354
6,Bidslab64,3679,0.529,0.472


# 10 Min Rolling

In [19]:
for name, df in [("Train", train), ("Validation", val)]:

    # 10-minute rolling heart rate
    df["hr_mean_roll20"] = df.groupby("segment")["hr_mean"].transform(
        lambda x: x.rolling(window=20, min_periods=1).mean()
    )

    # 10-minute rolling movement
    df["motion_mean_roll20"] = df.groupby("segment")["motion_mean"].transform(
        lambda x: x.rolling(window=20, min_periods=1).mean()
    )

    print(f"\n{name}")
    print("Missing HR values:", df["hr_mean_roll20"].isna().sum())
    print("Missing Motion values:", df["motion_mean_roll20"].isna().sum())

display(train[[
    "epoch",
    "hr_mean",
    "hr_mean_roll5",
    "hr_mean_roll20"
]].head(8))


Train
Missing HR values: 0
Missing Motion values: 0

Validation
Missing HR values: 0
Missing Motion values: 0


,epoch,hr_mean,hr_mean_roll5,hr_mean_roll20
0,0,64.333333,64.333333,64.333333
1,1,69.333333,66.833333,66.833333
2,2,74.000000,69.222222,69.222222
3,3,76.666667,71.083333,71.083333
4,4,72.000000,71.266667,71.266667
5,5,70.000000,72.400000,71.055556
6,6,69.600000,72.453333,70.847619
7,7,67.500000,71.153333,70.429167


In [20]:
for name, df in [("Train", train), ("Validation", val)]:

    # Heart-rate variability over 20 epochs
    df["hr_mean_roll_std20"] = df.groupby("segment")["hr_mean"].transform(
        lambda x: x.rolling(window=20, min_periods=2).std()
    ).fillna(0)

    # Movement variability over 20 epochs
    df["motion_mean_roll_std20"] = df.groupby("segment")["motion_mean"].transform(
        lambda x: x.rolling(window=20, min_periods=2).std()
    ).fillna(0)

    print(f"\n{name}")
    print("HR missing:", df["hr_mean_roll_std20"].isna().sum())
    print("Motion missing:", df["motion_mean_roll_std20"].isna().sum())

# Check first 8 rows
display(train[[
    "epoch",
    "hr_mean_roll_std5",
    "hr_mean_roll_std20",
    "motion_mean_roll_std5",
    "motion_mean_roll_std20"
]].head(8))


Train
HR missing: 0
Motion missing: 0

Validation
HR missing: 0
Motion missing: 0


,epoch,hr_mean_roll_std5,hr_mean_roll_std20,motion_mean_roll_std5,motion_mean_roll_std20
0,0,0.000000,0.000000,0.000000,0.000000
1,1,3.535534,3.535534,0.008181,0.008181
2,2,4.834291,4.834291,0.022578,0.022578
3,3,5.425420,5.425420,0.023336,0.023336
4,4,4.716402,4.716402,0.020224,0.020224
5,5,3.003701,4.250054,0.014808,0.018141
6,6,2.937270,3.918563,0.014175,0.017141
7,7,3.471439,3.816066,0.014331,0.016498


In [21]:
# New 10-minute features
new_features = [
    "hr_mean_roll20",
    "motion_mean_roll20",
    "hr_mean_roll_std20",
    "motion_mean_roll_std20"
]

# Combine with our previous 14 features
features_17 = features + new_features

# Prepare training and validation
X_train_17 = train[features_17]
X_val_17 = val[features_17]

y_train = train["stage_4"]
y_val = val["stage_4"]

# Verify
print("Training:", X_train_17.shape)
print("Validation:", X_val_17.shape)

print("Missing Train:", X_train_17.isna().sum().sum())
print("Missing Validation:", X_val_17.isna().sum().sum())

Training: (139780, 17)
Validation: (25676, 17)
Missing Train: 0
Missing Validation: 0


In [22]:
from sklearn.preprocessing import StandardScaler

# Create new scaler
scaler_17 = StandardScaler()

# Fit only on training data
X_train_17_scaled = scaler_17.fit_transform(X_train_17)

# Apply same transformation to validation
X_val_17_scaled = scaler_17.transform(X_val_17)

# Verify
print("Train:", X_train_17_scaled.shape)
print("Validation:", X_val_17_scaled.shape)

print("\nTrain means:")
print(X_train_17_scaled.mean(axis=0).round(2))

print("\nTrain standard deviations:")
print(X_train_17_scaled.std(axis=0).round(2))

Train: (139780, 17)
Validation: (25676, 17)

Train means:
[ 0. -0. -0.  0. -0. -0. -0. -0.  0. -0. -0. -0.  0. -0. -0.  0.  0.]

Train standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [24]:
from keras import Sequential, Input
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.utils import set_random_seed

set_random_seed(42)

# Build new model
model_17 = Sequential([
    Input(shape=(17,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(4, activation="softmax")
])

# Compile
model_17.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_17.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,364 (13.14 KB)

 Trainable params: 3,364 (13.14 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_17 = model_17.fit(
    X_train_17_scaled,
    y_train,
    validation_data=(X_val_17_scaled, y_val),
    epochs=100,
    batch_size=256,
    class_weight=soft_weights,
    callbacks=[early_stop]
)

Epoch 1/100


I0000 00:00:1794330235.703792   35788 subprocess_compilation.cc:349] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_22', 32 bytes spill stores, 32 bytes spill loads



547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4620 - loss: 1.1077 - val_accuracy: 0.4751 - val_loss: 1.1011
Epoch 2/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4927 - loss: 1.0624 - val_accuracy: 0.4803 - val_loss: 1.0904
Epoch 3/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5025 - loss: 1.0504 - val_accuracy: 0.4832 - val_loss: 1.0828
Epoch 4/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5062 - loss: 1.0434 - val_accuracy: 0.4892 - val_loss: 1.0754
Epoch 5/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5106 - loss: 1.0387 - val_accuracy: 0.4940 - val_loss: 1.0694
Epoch 6/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5122 - loss: 1.0337 - val_accuracy: 0.4974 - val_loss: 1.0644
Epoch 7/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5173 - loss: 1.0293 - val_accuracy: 0.5004 - val_loss: 1.0635
Epoch 8/100
547/547 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5184 - loss: 1.0266 - val_accuracy: 0.4979

In [26]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Predict validation data
y_prob_17 = model_17.predict(X_val_17_scaled)
y_pred_17 = np.argmax(y_prob_17, axis=1)

# Calculate metrics
accuracy_17 = accuracy_score(y_val, y_pred_17)
f1_17 = f1_score(y_val, y_pred_17, average="macro")

print("Validation Accuracy:", round(accuracy_17 * 100, 2), "%")
print("Macro F1:", round(f1_17, 4))

print("\nClassification Report:")
print(classification_report(
    y_val,
    y_pred_17,
    target_names=["Wake", "Light", "Deep", "REM"]
))

803/803 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Validation Accuracy: 51.06 %
Macro F1: 0.5017

Classification Report:
              precision    recall  f1-score   support

        Wake       0.47      0.48      0.48      2760
       Light       0.54      0.53      0.54     11731
        Deep       0.50      0.57      0.54      4687
         REM       0.47      0.45      0.46      6498

    accuracy                           0.51     25676
   macro avg       0.50      0.51      0.50     25676
weighted avg       0.51      0.51      0.51     25676



In [27]:
import joblib

# Save model
model_17.save("sleep_nn_17.keras")

# Save scaler
joblib.dump(scaler_17, "sleep_scaler_17.pkl")

# Save feature names and order
joblib.dump(features_17, "sleep_features_17.pkl")

print("18-feature model saved successfully!")

18-feature model saved successfully!


In [28]:
train.to_csv("train_temporal.csv", index=False)
val.to_csv("val_temporal.csv", index=False)

print("Temporal datasets saved!")

Temporal datasets saved!
